In [1]:
from pathlib import Path
import os


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    return start


def load_env_file() -> None:
    repo_root = _find_repo_root(Path.cwd())
    env_path = repo_root / ".env"
    example_path = repo_root / ".env.example"
    target = env_path if env_path.exists() else example_path
    if not target.exists():
        raise FileNotFoundError(
            f"Expected either {env_path} or {example_path} to exist."
        )

    with target.open() as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key, value)

    print(f"Loaded environment variables from {target.relative_to(repo_root)}")


load_env_file()


Loaded environment variables from .env


In [2]:
from langchain.agents import create_agent

/Users/thory/miniconda3/envs/tiny-ml-pt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_weather(city: str) -> str:
    """Get weather for a given city"""
    return f"Its always sunny in {city}"

agent = create_agent(
    model = 'openai:gpt-4o-mini',
    tools = [get_weather],
    prompt = 'You are a helpful assistant',
)

In [4]:
agent.invoke(
    {"message": [{"role":"user", "content": "what is the weather in sf"}]}
)

{'messages': [AIMessage(content='How can I assist you today? If you need information or help with something, feel free to ask!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 46, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CQXVuuZDajwXNZnljAIpba4ulYUFC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--65767475-7d88-49f8-85af-d1779820a2d5-0', usage_metadata={'input_tokens': 46, 'output_tokens': 22, 'total_tokens': 68, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

## Weather Forecasting Agent

In [5]:
# Step 1: System Prompt - The Agent's initial instructions or personality.
system_prompt = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean whereever they are, use the get_user_location tool to find their location."""

In [6]:
# Step 2: Create tools - tools are functions that can be called, they interact with external data to get stuff done.
from langchain_core.tools import tool
import random

def get_weather_for_location(city: str) -> str:
    '''Get weather for a given city'''
    conditions = random.choice(['sunny', 'rainy', 'cloudy'])
    return f'It is {conditions} in {city}'

from langchain_core.runnables import RunnableConfig

# A lookup table for demo purposes
USER_LOCATION = {
    "1":"Florida",
    "2":"SF"
}

'''
@tool decorator turns Python callables into LangChain `StructuredTool` objects
that the agent can discover and invoke. It can then use LangChain's tool metadata 
like names, descriptions, config injections.
'''
@tool
def get_user_location(config: RunnableConfig) -> str:
    '''Retrieve user information'''
    user_id = config.get("configurable", {}).get("user_id")
    return USER_LOCATION[user_id]


In [7]:
# Step 3: Configure the model
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0,
)

In [8]:
# Step 4: Define response format
from dataclasses import dataclass

@dataclass
class WeatherResponse:
    conditions: str
    punny_response: str

In [9]:
# Step 5: Add memory for the agent to remember conversation history
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [10]:
# Step 6: Bring it all together
agent = create_agent(
    model=model,
    prompt=system_prompt,
    tools=[get_user_location, get_weather_for_location],
    response_format=WeatherResponse,
    checkpointer=checkpointer
)

# config = {"configurable": {"thread_id": "1"}}
# context = {"user_id": "1"}

'''
`config` is the run metadata shared across every runnable (models, tools, graphs).
`config` has reserved keys like "configurable", "run_name", "tags", "metadata", "callbacks".
"configurable" is a catch-all for values we want to read back inside the graph or tools.
"thread_id" must be supplied when using `InMemorySaver` or any checkpointer - it decides which conversation thread to load.
'''

config = {"configurable": {"thread_id": "1", "user_id": "2"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
)

response['structured_response']

WeatherResponse(conditions='rainy', punny_response="Looks like it's a wet and wild day in SF! Don't forget your umbrella, or you might just get caught in a drizzle of disappointment!")

**More control over the model using provider's package**

```python
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5",
    temperature=0.1,
    max_tokens=1000,
    timeout=30
)
agent = create_agent(model, tools=tools)
```

In [11]:

response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config
)

response['structured_response']

WeatherResponse(conditions='clear', punny_response="You're welcome! I'm always here to brighten your day, even when the weather is clear!")

# More About Agents

In [12]:
# Dynamic Model Loading
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langgraph.runtime import Runtime

def select_model(state: AgentState, runtime: Runtime) -> ChatOpenAI:
    '''Chooses model based on conversation complexity'''
    messages = state['messages']
    message_count = len(messages)
    print(messages)

    if message_count < 10:
        return ChatOpenAI(model='gpt-4.1-mini').bind_tools(tools)
    else:
        return ChatOpenAI(model='gpt-5').bind_tools(tools)
    
tools = [get_user_location, get_weather_for_location]

# Pass this function as the model in `create_agent()`
agent = create_agent(select_model, tools=tools)
agent.invoke({"messages":[{"role": "user", "content": "what is the weather outside?"}]},
             config=config)

[HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='7f6a721c-c122-4f52-8b14-803700b53613')]
[HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='7f6a721c-c122-4f52-8b14-803700b53613'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_3a602tVV1qDQllXEkzFiKdw3', 'function': {'arguments': '{}', 'name': 'get_user_location'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 65, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c064fdde7c', 'id': 'chatcmpl-CQXWiYZUzOzq8KRFDRpm96juzHOG1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='ru

{'messages': [HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='7f6a721c-c122-4f52-8b14-803700b53613'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_3a602tVV1qDQllXEkzFiKdw3', 'function': {'arguments': '{}', 'name': 'get_user_location'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 65, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c064fdde7c', 'id': 'chatcmpl-CQXWiYZUzOzq8KRFDRpm96juzHOG1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--d4cc7827-0368-4ce1-881a-28ed258cee42-0', tool_calls=[{'name': 'get_user_location', 'args': {}, 'id': 'call_3a602tVV1qDQllXEk

In [13]:
# ToolNode: Creating tools like before (callables, @tool, provider's dict) will internally create
# a ToolNode. Here, you can defined ToolNode yourself for finer control.
from langchain_core.tools import tool
from langchain.agents import ToolNode
from langchain.agents import create_agent

tool_node = ToolNode(
    tools = [get_user_location, get_weather_for_location],
    handle_tool_errors = 'check again if error'
)

'''
This should not have run without passing `config` but since we handled errors
using `handle_tool_errors`, it bypasses the error.
'''
agent = create_agent(model, tools=tool_node)
result = agent.invoke({"messages":[{"role": "user", "content": "what is the weather outside?"}]})
result

{'messages': [HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='29adad9e-fb6d-4965-b3c8-63b7fad82206'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_IHmMr9EaJsAYpEUGgxirCCE8', 'function': {'arguments': '{}', 'name': 'get_user_location'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 65, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CQXX5oxfZ8d2ggfyzvVJD3rRqTncy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--1eaee805-4dd7-41fd-a431-5cc6daa42469-0', tool_calls=[{'name': 'get_user_location', 'args': {}, 'id': 'call_IHmMr9EaJsAYpEUGgx

In [33]:
model = ChatOpenAI(
    model='gpt-4.1-mini'
)

* `langchain.agents.AgentState`: TypedDict used to track what agents knows so far (conversation messages, intermediate tool steps, next node to execute).
* `langchain.agents.middleware.types.ModelRequest`: a dataclass LangGraph emits whenever the agents needs model call. Carries prompt messages plus runtime config.
* `langgraph.runtime.Runtime`: event scheduler that drivers a LangGraph.

In [ ]:
# Dynamic Prompt with Middleware
from typing import TypedDict

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware.types import ModelRequest, modify_model_request
from langgraph.runtime import Runtime

class Context(TypedDict):
    user_role: str
'''
Can't use config = {"context":"..."} here because Middlewares registered with
@modify_model_request can only be populated by context= argument in agent.invoke;

Use `config` dict when working with plain LangChain runnables, tools, model wrappers etc.
These will pull values from RunnableConfig. Place custom values under the `configurable` key.

Use `context=` argument in agent.invoke when using LangGraph's run-scoped state
like middlewares, graph nodes that accept runtime, store helpers etc. They will
look at runtime.context.
'''


@modify_model_request
def dynamic_system_prompt(state: AgentState, request: ModelRequest, runtime: Runtime[Context]) -> ModelRequest:
    print(runtime.context)
    user_role = runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        prompt = f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        prompt = f"{base_prompt} Explain concepts simply and avoid jargon."
    else:
        prompt = base_prompt
    print(request)
    request['system_prompt'] = prompt
    # Can't do `request.system_prompt=` here because the request dict might not 
    # have the system_prompt key yet.
    return request

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=tools,
    middleware=[dynamic_system_prompt],
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    # {"context": {"user_role": "expert"}}
    context=Context(user_role='expert')
)

{'user_role': 'expert'}
{'messages': [HumanMessage(content='Explain machine learning', additional_kwargs={}, response_metadata={}, id='dbf83858-4296-4c49-8ddc-313a9ac548f5')]}


In [31]:
print(result['messages'][1].content)

Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. Instead of being programmed to perform specific tasks, machine learning systems learn from data and improve their performance over time.

Here are some key concepts and components of machine learning:

1. **Data**: Machine learning relies heavily on data, which can be in various forms such as text, images, audio, or numerical values. This data is used to train machine learning models.

2. **Training**: During the training phase, a machine learning model is exposed to a dataset that includes input-output pairs. The model learns to recognize patterns in the data, adjusting its internal parameters to minimize errors in predictions.

3. **Models**: A model is a mathematical representation of the relationship between input features and output predictions. Different types of models can be used d

In [34]:
from typing import TypedDict
from typing_extensions import Annotated
from langgraph.graph.message import add_messages
from langchain.agents import create_agent
from langchain.agents import AgentState

class CustomAgentState(AgentState):
    messages: Annotated[list, add_messages]
    user_preferences: dict

agent = create_agent(
    model,
    tools=tools,
    state_schema=CustomAgentState
)

# The agent can now track additional state beyond messages. This custom state can be accessed and updated throughout the conversation.
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "OneWord"},
})

In [36]:
result

{'messages': [HumanMessage(content='I prefer technical explanations', additional_kwargs={}, response_metadata={}, id='963b1d48-8bb1-4469-b9b0-c25e22db5266'),
  AIMessage(content="Sure! Please specify the topic or concept you'd like a technical explanation about, and I'll provide a detailed response.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 63, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CNegPeQQ8KeN1cQ4CuJrgHOGlImLJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--8e03bed1-d73b-47a8-babd-aaf312e06de0-0', usage_metadata={'input_tokens': 63, 'output_tokens': 23, 'total_tokens': 86, 'input_token_details': {'aud

In [19]:
from pydantic import BaseModel
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str
    notes: str

_CONTACT_DIRECTORY = {
    "John Doe": {
        "email": "john@example.com",
        "phone": "(555) 123-4567",
        "notes": "Preferred contact window: 9am–2pm PST.",
    },
    "Jane Smith": {
        "email": "jane@contoso.ai",
        "phone": "(555) 987-6543",
        "notes": "Product lead for enterprise accounts.",
    },
}

@tool
def get_intent(query: str) -> str:
    '''Gets user intent and returns what they are seeking'''
    pass

@tool
def search_tool(query: str) -> str:
    """Resolve contact details from the external directory by fuzzy name lookup."""
    normalized = query.strip().lower()
    for name, record in _CONTACT_DIRECTORY.items():
        if normalized in name.lower():
            return (
                f"name={name}; "
                f"email={record['email']}; "
                f"phone={record['phone']}; "
                f"notes={record['notes']}; "
            )
    return "Contact not found in external directory."


agent = create_agent(
    model,
    tools=[search_tool],
    response_format=ContactInfo,
    prompt='You have two tools: use `search_tool` to search for the contact details in directory. Use `get_intent`. for finding out what piece of information is required by the user.'
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Details of user with phone number (555) 123-4567"}]
})

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

KeyboardInterrupt: 

In [ ]:
# Custom Agent State
from typing import TypedDict
from typing_extensions import Annotated
from langgraph.graph.message import add_messages
from langchain.agents import create_agent
from langchain.agents import AgentState

class CustomAgentState(AgentState):
    messages: Annotated[list, add_messages]
    user_preferences: dict

agent = create_agent(
    model,
    tools=tools,
    state_schema=CustomAgentState
)

# The agent can now track additional state beyond messages. This custom state can be accessed and updated throughout the conversation.
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

In [ ]:
# Pre-model Hook
from langchain_core.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langchain.agents import create_agent

def trim_messages(state):
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return {"messages": messages}

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }

agent = create_agent(
    model,
    tools=tools,
    pre_model_hook=trim_messages
)

In [ ]:
# Post-model Hook
from langchain_core.messages import AIMessage, RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES

def validate_response(state):
    """Check model response for policy violations."""
    messages = state["messages"]
    last_message = messages[-1]

    if "confidential" in last_message.content.lower():
        return {
            "messages": [
                RemoveMessage(id=REMOVE_ALL_MESSAGES),
                *messages[:-1],
                AIMessage(content="I cannot share confidential information.")
            ]
        }

    return {}

agent = create_agent(
    model,
    tools=tools,
    post_model_hook=validate_response
)

In [ ]:
# Streaming : If the agent executes multiple steps, show intermediate progress with this
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]
}, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")